# Section 03 lab — DataFrames and data quality

Every exercise reads the source again, so it can run independently.


## Exercise 1 — Select, filter and derive columns


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
transaction_schema = StructType([
    StructField('transaction_id', StringType(), True),
    StructField('transaction_date', StringType(), True),
    StructField('customer_id', StringType(), True),
    StructField('product_id', StringType(), True),
    StructField('quantity', StringType(), True),
    StructField('unit_price', StringType(), True),
    StructField('status', StringType(), True),
])
raw = spark.read.schema(transaction_schema).option('header', True).csv('../data/transactions.csv')
completed = (raw.filter(F.col('status') == 'COMPLETED')
             .select('transaction_id', 'customer_id', 'product_id', 'quantity', 'unit_price'))
completed.withColumn('source_row', F.lit('completed transaction')).show(10, truncate=False)
print('Completed source rows:', completed.count())


## Exercise 2 — Remove duplicates and convert types safely


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
transaction_schema = StructType([
    StructField('transaction_id', StringType(), True),
    StructField('transaction_date', StringType(), True),
    StructField('customer_id', StringType(), True),
    StructField('product_id', StringType(), True),
    StructField('quantity', StringType(), True),
    StructField('unit_price', StringType(), True),
    StructField('status', StringType(), True),
])
raw = spark.read.schema(transaction_schema).option('header', True).csv('../data/transactions.csv')
typed = (raw.dropDuplicates(['transaction_id'])
         .withColumn('order_date', F.expr('try_cast(transaction_date as date)'))
         .withColumn('qty', F.expr('try_cast(quantity as int)'))
         .withColumn('price', F.expr('try_cast(unit_price as decimal(12,2))')))
typed.select('transaction_id', 'order_date', 'qty', 'price').show(15)
print('Raw:', raw.count(), 'Unique:', typed.count())
assert raw.count() == 58 and typed.count() == 57


## Exercise 3 — Split valid and rejected records with reasons


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
transaction_schema = StructType([
    StructField('transaction_id', StringType(), True),
    StructField('transaction_date', StringType(), True),
    StructField('customer_id', StringType(), True),
    StructField('product_id', StringType(), True),
    StructField('quantity', StringType(), True),
    StructField('unit_price', StringType(), True),
    StructField('status', StringType(), True),
])
raw = spark.read.schema(transaction_schema).option('header', True).csv('../data/transactions.csv')
typed = (raw.dropDuplicates(['transaction_id'])
         .withColumn('order_date', F.expr('try_cast(transaction_date as date)'))
         .withColumn('qty', F.expr('try_cast(quantity as int)'))
         .withColumn('price', F.expr('try_cast(unit_price as decimal(12,2))')))
reason = F.concat_ws('; ',
    F.when(F.col('order_date').isNull(), 'invalid date'),
    F.when(F.col('qty').isNull() | (F.col('qty') <= 0), 'invalid quantity'),
    F.when(F.col('price').isNull() | (F.col('price') < 0), 'invalid price'),
    F.when(F.col('customer_id').isNull() | (F.length(F.trim('customer_id')) == 0), 'missing customer'),
    F.when(F.col('product_id').isNull() | (F.length(F.trim('product_id')) == 0), 'missing product'))
classified = typed.withColumn('reject_reason', reason)
valid = (classified.filter(F.col('reject_reason') == '')
         .withColumn('line_total', F.col('qty') * F.col('price')))
rejected = classified.filter(F.col('reject_reason') != '')
rejected.select('transaction_id', 'reject_reason').orderBy('transaction_id').show(truncate=False)
print('Valid:', valid.count(), 'Rejected:', rejected.count())
assert valid.count() == 50 and rejected.count() == 7


## Exercise 4 — Build a status summary from a fresh read


In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master('local[2]')
         .appName('PySpark-Course')
         .config('spark.sql.shuffle.partitions', '4')
         .config('spark.sql.adaptive.enabled', 'true')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
transaction_schema = StructType([
    StructField('transaction_id', StringType(), True),
    StructField('transaction_date', StringType(), True),
    StructField('customer_id', StringType(), True),
    StructField('product_id', StringType(), True),
    StructField('quantity', StringType(), True),
    StructField('unit_price', StringType(), True),
    StructField('status', StringType(), True),
])
raw = spark.read.schema(transaction_schema).option('header', True).csv('../data/transactions.csv')
summary = (raw.groupBy(F.upper(F.trim('status')).alias('status'))
           .agg(F.count('*').alias('rows'))
           .orderBy('status'))
summary.show()
assert summary.agg(F.sum('rows')).first()[0] == 58
